<a href="https://colab.research.google.com/github/codematser69/candidate-ranking--dashboard/blob/main/panda_tech.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# CELL 1: Mount Google Drive
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

# Confirm your folder name matches EXACTLY (case + hyphen/underscore sensitive)
# CONFIRMED real folder name: redrob_hackathon (underscore, not hyphen)
import os
print(os.listdir('/content/drive/MyDrive/redrob_hackathon'))


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
['candidates.jsonl', 'job_description.docx', 'sample_candidates.json', 'validate_submission.py', 'submission_spec.docx', 'submission_metadata_template.yaml', 'candidate_schema.json', 'redrob_signals_doc.docx', 'sample_submission.csv', 'output']


In [ ]:
# ============================================================
# CELL 2: Install only what's missing on Colab
# ============================================================
!pip install -q lightgbm sentence-transformers
print("Libraries installed")


Libraries installed


In [ ]:
# ============================================================
# CELL 3: Imports & Config
# ============================================================
import json
import os
import re
import math
import time
import warnings
from datetime import datetime
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
warnings.filterwarnings("ignore")

# PATHS — CONFIRMED real folder name: redrob_hackathon (underscore, not hyphen)
DRIVE_BASE = "/content/drive/MyDrive/redrob_hackathon"
CANDIDATES_PATH = f"{DRIVE_BASE}/candidates.jsonl"     # NOTE: .jsonl, not .jsonl.gz
OUTPUT_DIR = f"{DRIVE_BASE}/output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Config ready")
print(f"Dataset path: {CANDIDATES_PATH}")
print(f"Output path : {OUTPUT_DIR}")
print(f"File exists : {os.path.exists(CANDIDATES_PATH)}")   # MUST print True before proceeding to Cell 4

Config ready
Dataset path: /content/drive/MyDrive/redrob_hackathon/candidates.jsonl
Output path : /content/drive/MyDrive/redrob_hackathon/output
File exists : True


In [ ]:
# ============================================================
# CELL 4: Load all 100K candidates (plain .jsonl, no gzip needed)
# ============================================================
def load_candidates(path):
    candidates = []
    with open(path, "r", encoding="utf-8") as f:
        for line in tqdm(f, desc="Loading candidates", unit=" lines"):
            line = line.strip()
            if line:
                candidates.append(json.loads(line))
    return candidates

start = time.time()
candidates = load_candidates(CANDIDATES_PATH)
print(f"Loaded {len(candidates):,} candidates in {time.time()-start:.1f}s")


Loading candidates: 100000 lines [00:17, 5792.16 lines/s]


Loaded 100,000 candidates in 18.4s


In [ ]:
# ============================================================
# CELL 5: Explore one candidate record — confirm structure matches this guide
# ============================================================
sample = candidates[0]
print(json.dumps(sample, indent=2, default=str)[:2000])


{
  "candidate_id": "CAND_0000001",
  "profile": {
    "anonymized_name": "Ira Vora",
    "headline": "Backend Engineer | SQL, Spark, Cloud",
    "summary": "Software / data professional with 6.9 years of experience building data pipelines, backend systems, and analytics infrastructure. I'm a backend/data hybrid \u2014 Spark, Airflow, SQL warehouses are home territory; I'm building competence on the ML side. My toolkit is solid on the data engineering side \u2014 Python, SQL, Spark, Airflow, warehouse design \u2014 and I've completed a couple of self-directed ML projects (Kaggle competitions, side projects fine-tuning small models). Interested in transitioning toward more AI/ML-focused work, ideally at a company where I can leverage my existing data-infra skills while learning modern ML practice.",
    "location": "Toronto",
    "country": "Canada",
    "years_of_experience": 6.9,
    "current_title": "Backend Engineer",
    "current_company": "Mindtree",
    "current_company_size": "1

In [ ]:
# CELL 6: Confirm top-level keys match what this guide expects
all_keys = set()
for c in candidates[:1000]:
    all_keys.update(c.keys())
print("Top-level keys found:", sorted(all_keys))
# Expected: candidate_id, profile, career_history, education,
#           skills, certifications, languages, redrob_signals

Top-level keys found: ['candidate_id', 'career_history', 'certifications', 'education', 'languages', 'profile', 'redrob_signals', 'skills']


In [ ]:
# CELL 7: Confirm redrob_signals keys (should be 23 signals)
print(json.dumps(candidates[0]["redrob_signals"], indent=2))

{
  "profile_completeness_score": 86.9,
  "signup_date": "2025-10-16",
  "last_active_date": "2026-05-20",
  "open_to_work_flag": true,
  "profile_views_received_30d": 23,
  "applications_submitted_30d": 2,
  "recruiter_response_rate": 0.34,
  "avg_response_time_hours": 177.8,
  "skill_assessment_scores": {
    "NLP": 38.8,
    "Image Classification": 64.8,
    "Fine-tuning LLMs": 41.6,
    "Speech Recognition": 53.7
  },
  "connection_count": 356,
  "endorsements_received": 35,
  "notice_period_days": 60,
  "expected_salary_range_inr_lpa": {
    "min": 18.7,
    "max": 36.1
  },
  "preferred_work_mode": "onsite",
  "willing_to_relocate": false,
  "github_activity_score": 9.2,
  "search_appearance_30d": 249,
  "saved_by_recruiters_30d": 4,
  "interview_completion_rate": 0.71,
  "offer_acceptance_rate": 0.58,
  "verified_email": true,
  "verified_phone": true,
  "linkedin_connected": false
}


In [ ]:
# ============================================================
# CELL 8: Flatten all candidates into a Pandas DataFrame
# Every field path below matches the REAL nested schema.
# ============================================================

def days_since(date_str, reference_date="2026-06-20"):
    """Compute days between a date string and the reference date (today)."""
    if not date_str:
        return 999
    try:
        d = datetime.strptime(date_str, "%Y-%m-%d")
        ref = datetime.strptime(reference_date, "%Y-%m-%d")
        return (ref - d).days
    except Exception:
        return 999

def get_best_education_tier(education_list):
    """Education tier lives PER DEGREE. Take the best (lowest number) tier found."""
    if not education_list:
        return 3  # default mid-tier if missing
    tiers = []
    for edu in education_list:
        t = edu.get("tier", "")
        match = re.search(r"(\d+)", str(t))
        if match:
            tiers.append(int(match.group(1)))
    return min(tiers) if tiers else 3

rows = []
for c in tqdm(candidates, desc="Flattening to DataFrame"):
    cid = c.get("candidate_id", "")
    profile = c.get("profile", {}) or {}
    signals = c.get("redrob_signals", {}) or {}
    career_history = c.get("career_history", []) or []
    education = c.get("education", []) or []
    skills = c.get("skills", []) or []

    row = {
        "candidate_id"        : cid,
        "current_title"       : profile.get("current_title", ""),
        "current_company"     : profile.get("current_company", ""),
        "current_industry"    : profile.get("current_industry", ""),
        "current_company_size": profile.get("current_company_size", ""),
        "location_city"       : profile.get("location", ""),
        "country"             : profile.get("country", ""),
        "total_exp_years"     : profile.get("years_of_experience", 0) or 0,
        "headline"            : profile.get("headline", ""),
        "summary"             : profile.get("summary", ""),

        # From redrob_signals, NOT top-level
        "notice_period_days"  : signals.get("notice_period_days", 999) or 999,
        "willing_to_relocate" : int(bool(signals.get("willing_to_relocate", False))),
        "open_to_work_flag"   : int(bool(signals.get("open_to_work_flag", False))),
        "last_active_days_ago": days_since(signals.get("last_active_date", "")),
        "recruiter_response_rate": signals.get("recruiter_response_rate", 0) or 0,
        "github_activity_score": signals.get("github_activity_score", 0) or 0,
        "profile_completeness_score": signals.get("profile_completeness_score", 0) or 0,
        "interview_completion_rate": signals.get("interview_completion_rate", 0) or 0,
        "offer_acceptance_rate": signals.get("offer_acceptance_rate", 0) or 0,

        # Education tier extracted from nested list
        "education_tier"      : get_best_education_tier(education),

        # Raw blobs kept for later inspection / reasoning generation
        "skills_raw"          : json.dumps(skills, default=str),
        "career_history_raw"  : json.dumps(career_history, default=str),
        "education_raw"       : json.dumps(education, default=str),
        "full_text"           : "",
    }

    # Pull EVERY redrob_signal into its own column (sig_ prefix), skipping nested dicts/dates
    for sig_key, sig_val in signals.items():
        if isinstance(sig_val, (int, float, bool)):
            row[f"sig_{sig_key}"] = float(sig_val)
        # skip dicts (skill_assessment_scores, expected_salary_range_inr_lpa) and date strings here —
        # handle those separately if you want them as features later

    # Build searchable full-text blob for TF-IDF / keyword matching
    skills_list = [s.get("name", "") for s in skills if isinstance(s, dict)]

    career_text = []
    for job in career_history:
        if isinstance(job, dict):
            career_text.append(job.get("title", ""))
            career_text.append(job.get("description", ""))

    education_text = []
    for edu in education:
        if isinstance(edu, dict):
            education_text.append(edu.get("degree", ""))
            education_text.append(edu.get("field_of_study", ""))
            education_text.append(edu.get("institution", ""))

    row["full_text"] = " ".join(filter(None, [
        row["current_title"],
        row["headline"],
        " ".join(skills_list),
        " ".join(career_text),
        " ".join(education_text),
        row["summary"],
    ])).lower()

    rows.append(row)

df = pd.DataFrame(rows)
print(f"DataFrame created: {df.shape}")

# Sanity checks
print(f"\nTotal rows         : {len(df):,}")
print(f"Unique IDs          : {df['candidate_id'].nunique():,}")
print(f"Experience range    : {df['total_exp_years'].min():.1f} - {df['total_exp_years'].max():.1f}")
print(f"Notice period range : {df['notice_period_days'].min()} - {df['notice_period_days'].max()}")
sig_cols = [c for c in df.columns if c.startswith("sig_")]
print(f"Behavioral signal columns: {len(sig_cols)}")

# Save checkpoint to Drive so you don't have to re-flatten 100K rows every session
df.to_parquet(f"{OUTPUT_DIR}/candidates_flat.parquet", index=False)
print(f"\nCheckpoint saved: candidates_flat.parquet")


Flattening to DataFrame: 100%|██████████| 100000/100000 [00:21<00:00, 4675.72it/s]


In [ ]:
df = pd.read_parquet(f"{OUTPUT_DIR}/candidates_flat.parquet")

chunk-2: Feature Engineering & Signal Extraction

In [ ]:
# ============================================================
# CELL 9: Paste the FULL job_description text here
# (Open job_description.docx, copy everything, paste below)
# ============================================================
JOB_DESCRIPTION = """
Job Description: Senior AI Engineer — Founding Team
Company: Redrob AI (Series A AI-native talent intelligence platform)
Location: Pune/Noida, India (Hybrid) | Open to relocation from Tier-1 Indian cities
Experience Required: 5-9 years

[... paste the ENTIRE job_description.docx text here, all sections,
     including "What you'd actually be doing", the skills inventory,
     the disqualifiers, and the "Final note for hackathon participants" section.
     Do not truncate it — every section carries scoring signal. ]
"""

# Required skills — derived directly from the JD's "Things you absolutely need" section
JD_REQUIRED_SKILLS = [
    "embeddings", "embedding", "sentence-transformers", "bge", "e5",
    "vector database", "pinecone", "weaviate", "qdrant", "milvus",
    "opensearch", "elasticsearch", "faiss", "hybrid search",
    "python", "ndcg", "mrr", "map", "a/b test", "ranking", "retrieval",
]

# Nice-to-have — from "Things we'd like you to have but won't reject you for"
JD_NICE_SKILLS = [
    "lora", "qlora", "peft", "fine-tuning", "learning-to-rank", "xgboost",
    "lightgbm", "hr-tech", "recruiting", "distributed systems",
    "inference optimization", "open source", "github",
]

# Explicit negative signals — "Things we explicitly do NOT want"
JD_DISQUALIFYING_COMPANIES = [
    "tcs", "infosys", "wipro", "accenture", "cognizant", "capgemini"
]
JD_CV_SPEECH_ROBOTICS_ONLY = ["computer vision", "speech recognition", "robotics"]

JD_PREFERRED_LOCATIONS = ["pune", "noida", "hyderabad", "mumbai", "delhi", "ncr"]
JD_EXP_MIN = 5
JD_EXP_MAX = 9
JD_NOTICE_IDEAL = 30   # JD explicitly prefers sub-30-day notice

print(f"JD loaded | Required: {len(JD_REQUIRED_SKILLS)} | Nice: {len(JD_NICE_SKILLS)}")

In [ ]:
# ============================================================
# CELL 10: TF-IDF cosine similarity — all candidates vs JD
# ============================================================
print("Building TF-IDF vectorizer...")
t0 = time.time()

corpus = df["full_text"].fillna("").tolist()
jd_text = JOB_DESCRIPTION.lower()

tfidf = TfidfVectorizer(
    max_features=50_000,
    ngram_range=(1, 2),
    min_df=3,
    sublinear_tf=True,
    strip_accents="unicode",
    analyzer="word",
    stop_words="english",
)
tfidf.fit(corpus + [jd_text])
candidate_matrix = tfidf.transform(corpus)
jd_vector = tfidf.transform([jd_text])

tfidf_scores = cosine_similarity(jd_vector, candidate_matrix).flatten()
df["score_tfidf"] = tfidf_scores

print(f"TF-IDF done in {time.time()-t0:.1f}s")
print(f"Score range: {tfidf_scores.min():.4f} - {tfidf_scores.max():.4f}")
print(f"Top-1% threshold: {np.percentile(tfidf_scores, 99):.4f}")


In [ ]:
# ============================================================
# CELL 11: Hard keyword matching (required + nice-to-have)
# ============================================================
def compute_keyword_score(text, required, nice):
    text = text.lower()
    req_hits = sum(1 for s in required if s in text)
    nice_hits = sum(1 for s in nice if s in text)
    return (
        req_hits / max(len(required), 1),
        nice_hits / max(len(nice), 1),
        req_hits,
    )

print("Computing keyword scores...")
kw_results = df["full_text"].apply(
    lambda t: compute_keyword_score(t, JD_REQUIRED_SKILLS, JD_NICE_SKILLS)
)
df["score_kw_required"] = kw_results.apply(lambda x: x[0])
df["score_kw_nice"] = kw_results.apply(lambda x: x[1])
df["kw_required_count"] = kw_results.apply(lambda x: x[2])
print(df[["score_kw_required", "score_kw_nice"]].describe().round(3))

In [ ]:
# ============================================================
# CELL 12: Experience fit (Gaussian-style penalty outside ideal range)
# ============================================================
def exp_fit(years, ideal_min, ideal_max):
    if years < 0:
        return 0.0
    if ideal_min <= years <= ideal_max:
        return 1.0
    gap = ideal_min - years if years < ideal_min else years - ideal_max
    return math.exp(-0.3 * gap)

df["score_exp_fit"] = df["total_exp_years"].apply(
    lambda y: exp_fit(float(y), JD_EXP_MIN, JD_EXP_MAX)
)
print(df["score_exp_fit"].describe().round(3))

In [ ]:
# ============================================================
# CELL 13: Recency — exponential decay by days since last active
# ============================================================
def recency_score(days_ago):
    return math.exp(-max(float(days_ago), 0) / 180)  # 180-day half-life, tune as needed

df["score_recency"] = df["last_active_days_ago"].apply(recency_score)
print(df["score_recency"].describe().round(3))

In [ ]:
# ============================================================
# CELL 14: Notice period — lower is better, JD wants sub-30-day
# ============================================================
def notice_score(days, ideal_max=30):
    days = max(float(days), 0)
    if days <= ideal_max:
        return 1.0
    return math.exp(-0.02 * (days - ideal_max))

df["score_notice"] = df["notice_period_days"].apply(
    lambda d: notice_score(d, JD_NOTICE_IDEAL)
)
print(df["score_notice"].describe().round(3))

In [ ]:
# ============================================================
# CELL 15: Location match (Pune/Noida preferred, India-wide acceptable,
# outside-India only scores well if willing_to_relocate)
# ============================================================
def location_score(city, country, preferred_locs, willing_to_relocate):
    city_l = str(city).lower()
    country_l = str(country).lower()

    if any(loc in city_l for loc in preferred_locs):
        return 1.0
    if country_l == "india":
        return 0.7  # other Indian city — acceptable per JD
    # Outside India: only case-by-case, no visa sponsorship — needs relocation willingness
    return 0.4 if willing_to_relocate else 0.1

df["score_location"] = df.apply(
    lambda r: location_score(
        r["location_city"], r["country"], JD_PREFERRED_LOCATIONS, bool(r["willing_to_relocate"])
    ),
    axis=1
)
print(df["score_location"].describe().round(3))

In [ ]:
# ============================================================
# CELL 16: Education tier (tier_1 = best, tier_5 = lowest; invert it)
# ============================================================
def edu_tier_score(tier):
    tier = max(1.0, min(5.0, float(tier)))
    return (6.0 - tier) / 5.0  # tier1 -> 1.0, tier5 -> 0.2

df["score_edu"] = df["education_tier"].apply(edu_tier_score)
print(df["score_edu"].describe().round(3))

In [ ]:
# ============================================================
# CELL 17: IT-services-only career penalty
# JD explicitly does NOT want candidates whose ENTIRE career is at
# TCS/Infosys/Wipro/Accenture/Cognizant/Capgemini with no product company exposure.
# Current employer alone is NOT enough — check the whole career history.
# ============================================================
def it_services_penalty(career_history_raw, current_industry):
    try:
        history = json.loads(career_history_raw)
    except Exception:
        history = []

    if not history:
        # fall back to current industry only
        return 0.5 if str(current_industry).lower() == "it services" else 1.0

    services_count = sum(
        1 for job in history
        if isinstance(job, dict) and any(
            comp in str(job.get("company", "")).lower() or
            str(job.get("industry", "")).lower() == "it services"
            for comp in JD_DISQUALIFYING_COMPANIES
        )
    )
    total_jobs = len(history)
    services_ratio = services_count / max(total_jobs, 1)

    if services_ratio >= 0.99:   # entire career at services firms
        return 0.3
    elif services_ratio >= 0.5:  # majority but not entire career
        return 0.7
    return 1.0

df["score_it_services_fit"] = df.apply(
    lambda r: it_services_penalty(r["career_history_raw"], r["current_industry"]),
    axis=1
)
print(df["score_it_services_fit"].describe().round(3))

In [ ]:
# ============================================================
# CELL 18: Aggregate redrob_signals into one score
# ============================================================
sig_cols = [c for c in df.columns if c.startswith("sig_")]
print(f"Found {len(sig_cols)} signal columns: {sig_cols}")

scaler = MinMaxScaler()
df[sig_cols] = df[sig_cols].fillna(0)
df_sig_norm = pd.DataFrame(
    scaler.fit_transform(df[sig_cols]),
    columns=[f"norm_{c}" for c in sig_cols],
    index=df.index
)
df = pd.concat([df, df_sig_norm], axis=1)
norm_sig_cols = [f"norm_{c}" for c in sig_cols]
df["score_signals"] = df[norm_sig_cols].mean(axis=1)

print(df["score_signals"].describe().round(3))

In [ ]:
# ============================================================
# CELL 19: Honeypot detection — flags impossible/contradictory profiles
# Uses the REAL nested career_history/education/skills structure.
# ============================================================
def detect_honeypot(c, total_exp_years):
    flags = 0
    career_history = c.get("career_history", []) or []
    education = c.get("education", []) or []
    skills = c.get("skills", []) or []

    # Flag 1: single job tenure > 15 years (suspicious for most roles)
    for job in career_history:
        if isinstance(job, dict):
            dur_months = job.get("duration_months", 0) or 0
            if dur_months > 15 * 12:
                flags += 1

    # Flag 2: "expert"/"advanced" proficiency with 0 months experience
    expert_zero = sum(
        1 for sk in skills
        if isinstance(sk, dict)
        and str(sk.get("proficiency", "")).lower() in ("expert", "advanced")
        and (sk.get("duration_months", 1) or 1) == 0
    )
    if expert_zero >= 3:
        flags += 2

    # Flag 3: total experience exceeds what's possible since graduation
    for edu in education:
        if isinstance(edu, dict):
            end_year = edu.get("end_year", None)
            if end_year:
                max_possible = 2026 - int(end_year)  # current year per this conversation
                if total_exp_years > max_possible + 2:  # 2-year buffer for internships/early starts
                    flags += 3

    # Flag 4: extreme skill count (padding signal)
    if len(skills) > 40:
        flags += 1

    return min(flags / 5.0, 1.0)

print("Running honeypot detection...")
t0 = time.time()
honeypot_scores = []
for i, c in enumerate(tqdm(candidates, desc="Honeypot check")):
    honeypot_scores.append(detect_honeypot(c, df.iloc[i]["total_exp_years"]))
df["honeypot_score"] = honeypot_scores

print(f"Done in {time.time()-t0:.1f}s | Flagged {(df['honeypot_score'] > 0.4).sum()} candidates")

In [ ]:
# ============================================================
# CELL 20: Weighted composite score
# Adjust weights AFTER manually inspecting top 50 (see Key Tuning Tips below)
# ============================================================
WEIGHTS = {
    "score_tfidf"          : 0.25,
    "score_kw_required"    : 0.15,
    "score_exp_fit"        : 0.12,
    "score_signals"        : 0.15,
    "score_it_services_fit": 0.10,
    "score_recency"        : 0.08,
    "score_kw_nice"        : 0.05,
    "score_notice"         : 0.05,
    "score_location"       : 0.04,
    "score_edu"            : 0.01,
}
assert abs(sum(WEIGHTS.values()) - 1.0) < 0.001, "Weights must sum to 1.0"

df["composite_score"] = sum(df[feat] * w for feat, w in WEIGHTS.items())

# Honeypot penalty — multiplicative, so a high honeypot score crushes the final score
df["composite_score"] = df["composite_score"] * (1 - df["honeypot_score"] * 0.9)

# Rank
df["rank"] = df["composite_score"].rank(ascending=False, method="first").astype(int)

print("Composite scores computed")

top10 = df.nlargest(10, "composite_score")[
    ["candidate_id", "current_title", "composite_score", "rank",
     "score_tfidf", "score_kw_required", "total_exp_years", "honeypot_score"]
]
print("\nTop 10 candidates:")
print(top10.to_string(index=False))

feature_cols = (
    ["candidate_id", "rank", "composite_score"]
    + list(WEIGHTS.keys())
    + ["honeypot_score", "total_exp_years", "notice_period_days",
       "location_city", "country", "current_title", "current_company",
       "current_industry", "full_text"]
)
df[feature_cols].to_parquet(f"{OUTPUT_DIR}/candidates_features.parquet", index=False)
print(f"\nFeature dataset saved: {df[feature_cols].shape}")